# 🦷 Bone Loss Dataset Construction Pipeline
## Dental AI — Phase 4: From Raw Annotations → Training-Ready Dataset

**Input:** 439 radiograph–expert mask aligned samples + 26,005 tooth bounding boxes 
**Output:** `processed_dataset_tufts/train/{0,1}/` and `val/{0,1}/` with 224×224 crops

---

### Critical Design Decisions
| Decision | Choice | Why |
|----------|--------|-----|
| Overlap metric | `mask_pixels / crop_area` | NOT IoU — mask blobs are tiny, IoU collapses to ~0 |
| Threshold | 0.005–0.02 (configurable) | Even genuine lesions occupy only 0.5–2% of crop |
| Crop expansion | 1.5× W, 2.0× H | Asymmetric — teeth are ~3× taller than wide |
| Bbox jitter | ±15% | Simulates YOLO noise at inference |
| Split level | Per-image, not per-crop | Prevents data leakage from jittered copies |
| Deciduous teeth | Flagged, not removed | Evaluate inclusion/exclusion after inspection |

---
## Cell 1 — Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import os
import random
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image as PILImage

# Prevent PIL DecompressionBomb error on large composite figures
PILImage.MAX_IMAGE_PIXELS = None

print("All imports loaded.")

---
## Cell 2 — Configuration

**Update `BASE` if your drive path is different.** Everything else derives from it.

In [ ]:
# =============================================================================
# PATHS — Matched to your Google Drive structure
# =============================================================================
#   tufts/ZIP/
#     Radiographs/Radiographs/   → 1.JPG, 100.JPG, ...
#     Segmentation/Segmentation/ → teeth_bbox.json, teeth_polygon.json
#     Expert/Expert/             → expert.json
#     Expert/Expert/mask/        → 1.JPG, 100.JPG, ...
# =============================================================================

BASE = "/content/drive/MyDrive/tufts/ZIP"

RADIO_DIR    = os.path.join(BASE, "Radiographs", "Radiographs")
BBOX_JSON    = os.path.join(BASE, "Segmentation", "Segmentation", "teeth_bbox.json")
EXPERT_JSON  = os.path.join(BASE, "Expert", "Expert", "expert.json")
EXPERT_MASK  = os.path.join(BASE, "Expert", "Expert", "mask")

# Output — saved to Drive so it persists across sessions
OUTPUT_DIR = "/content/drive/MyDrive/tufts/processed_dataset_tufts"

# --- Crop parameters ---
EXPAND_W   = 1.5          # Width expansion factor
EXPAND_H   = 2.0          # Height expansion factor (more vertical — teeth are tall)
CROP_SIZE  = (224, 224)

# --- Label assignment ---
# CRITICAL: pixel-overlap ratio, NOT IoU
# overlap = mask_pixels_inside_crop / crop_area
OVERLAP_THRESHOLD = 0.01  # 1% — tune after visual inspection in Cell 9

# --- Bbox jitter for YOLO noise simulation ---
JITTER_FRACTION = 0.15    # ±15%

# --- Train/val split ---
VAL_RATIO   = 0.20
RANDOM_SEED = 42

# --- Validation ---
N_VISUAL_SAMPLES = 30

# --- Bbox filter ---
MIN_BBOX_DIM = 3          # Min width/height in pixels

# --- Deciduous teeth IDs ---
DECIDUOUS_IDS = set("ABCDEFGHIJKLMNOPQRST")

# --- Verify paths exist ---
for name, path in [("Radiographs", RADIO_DIR), ("Bbox JSON", BBOX_JSON),
                    ("Expert JSON", EXPERT_JSON), ("Expert Masks", EXPERT_MASK)]:
    exists = os.path.exists(path)
    status = "✓" if exists else "✗ NOT FOUND"
    print(f"  {status}  {name}: {path}")

print(f"\n  Output dir: {OUTPUT_DIR}")

---
## Cell 3 — Load Expert JSON

**Bug fix applied:** `expert.json` is a LIST of records, not a dict.  
Join key = `Path(item['External ID']).stem`, NOT a dict key.

In [ ]:
def load_expert_json(path):
    """
    Parse expert.json using External ID as join key.
    Returns DataFrame with columns: [stem, description]
    """
    with open(path, 'r') as f:
        data = json.load(f)

    records = []
    for item in data:
        stem = Path(item["External ID"]).stem
        description = item.get("Description", "")
        records.append({"stem": stem, "description": description})

    df = pd.DataFrame(records)
    print(f"[Expert] Loaded {len(df)} records, {df['stem'].nunique()} unique stems")
    
    # Quick check: show a few descriptions
    normal_count = df['description'].str.lower().str.contains('within normal limits').sum()
    print(f"  'Within normal limits': {normal_count}")
    print(f"  Pathological (other):   {len(df) - normal_count}")
    
    return df

df_expert = load_expert_json(EXPERT_JSON)
df_expert.head(3)

---
## Cell 4 — Load & Convert Bounding Boxes

**Bug fix applied:** Raw format is `[y1, x1, y2, x2]` corner points.  
Must convert to `[x, y, w, h]` with x/y swap.

In [ ]:
def load_bbox_json(path):
    """
    Parse teeth_bbox.json with CORRECT coordinate conversion.
    Raw: [y1, x1, y2, x2] → Converted: [x, y, w, h]
    """
    with open(path, 'r') as f:
        data = json.load(f)

    records = []
    for item in data:
        stem = Path(item["External ID"]).stem

        if "Label" not in item or "objects" not in item["Label"]:
            continue

        for obj in item["Label"]["objects"]:
            tooth_id = str(obj.get("title", "unknown")).strip()
            bbox_raw = obj.get("bounding box", None)

            if bbox_raw is None or len(bbox_raw) < 4:
                continue

            # CRITICAL conversion: [y1, x1, y2, x2] → [x, y, w, h]
            y1, x1, y2, x2 = bbox_raw
            x = x1
            y = y1
            w = x2 - x1
            h = y2 - y1

            records.append({
                "stem": stem,
                "tooth_id": tooth_id,
                "x": x, "y": y, "w": w, "h": h,
            })

    df = pd.DataFrame(records)
    print(f"[Bbox] Loaded {len(df)} tooth records from {df['stem'].nunique()} images")
    print(f"  Width  range: {df['w'].min():.0f} – {df['w'].max():.0f} px (mean {df['w'].mean():.1f})")
    print(f"  Height range: {df['h'].min():.0f} – {df['h'].max():.0f} px (mean {df['h'].mean():.1f})")
    print(f"  Avg teeth/image: {len(df) / df['stem'].nunique():.1f}")

    # Flag tooth types
    deciduous = df['tooth_id'].isin(DECIDUOUS_IDS)
    print(f"  Deciduous teeth (A-T): {deciduous.sum()}")
    print(f"  Degenerate (w or h < {MIN_BBOX_DIM}): {((df['w'] < MIN_BBOX_DIM) | (df['h'] < MIN_BBOX_DIM)).sum()}")

    return df

df_bbox = load_bbox_json(BBOX_JSON)
df_bbox.head(3)

---
## Cell 5 — Classify Expert Masks (Empty vs Annotated)

Empty mask (all-zero) = normal → all tooth crops from this image are negative  
Non-empty mask = pathological → per-crop overlap check needed

In [ ]:
def classify_expert_masks(mask_dir):
    """
    Load all expert masks. Classify as empty (normal) or annotated (pathological).
    
    VERIFIED: 285 empty masks ↔ 'within normal limits' descriptions
              154 annotated masks ↔ pathology descriptions
              Perfect label-mask consistency.
    """
    records = []
    mask_files = sorted(Path(mask_dir).glob("*"))

    for mf in mask_files:
        stem = mf.stem
        try:
            mask = cv2.imread(str(mf), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                print(f"  [WARN] Could not read: {mf.name}")
                continue

            is_empty = (mask.max() == 0)
            nonzero_count = int(np.count_nonzero(mask))

            records.append({
                "stem": stem,
                "mask_path": str(mf),
                "mask_empty": is_empty,
                "mask_nonzero_pixels": nonzero_count,
            })
        except Exception as e:
            print(f"  [ERROR] {mf.name}: {e}")

    df = pd.DataFrame(records)
    n_empty = df['mask_empty'].sum()
    n_annotated = (~df['mask_empty']).sum()
    print(f"[Masks] Total: {len(df)} | Empty (normal): {n_empty} | Annotated (pathological): {n_annotated}")

    return df

df_masks = classify_expert_masks(EXPERT_MASK)
df_masks.head(3)

---
## Cell 6 — Join All Sources & Clean Bboxes

Only images with **all three** annotation types (expert label + bbox + expert mask) are kept.  
Expected: ~423 fully annotated images.

In [ ]:
# ---- JOIN ----
mask_stems   = set(df_masks['stem'])
bbox_stems   = set(df_bbox['stem'])
expert_stems = set(df_expert['stem'])
valid_stems  = mask_stems & bbox_stems & expert_stems

print("[Join] Source counts:")
print(f"  Expert labels:  {len(expert_stems)}")
print(f"  Bounding boxes: {len(bbox_stems)}")
print(f"  Expert masks:   {len(mask_stems)}")
print(f"  FULL overlap:   {len(valid_stems)}")

# Filter bbox to valid stems only
df_bbox_valid = df_bbox[df_bbox['stem'].isin(valid_stems)].copy()

# Merge mask info
df_merged = df_bbox_valid.merge(
    df_masks[['stem', 'mask_path', 'mask_empty', 'mask_nonzero_pixels']],
    on='stem', how='inner'
)

# Merge expert description
df_merged = df_merged.merge(
    df_expert[['stem', 'description']],
    on='stem', how='inner'
)

print(f"  Merged tooth-level records: {len(df_merged)}")
print(f"  From {df_merged['stem'].nunique()} unique images")

In [ ]:
# ---- CLEAN BBOXES ----
n_before = len(df_merged)

# Remove degenerate bboxes (w or h < MIN_BBOX_DIM)
df_merged = df_merged[(df_merged['w'] >= MIN_BBOX_DIM) & (df_merged['h'] >= MIN_BBOX_DIM)].copy()
n_degenerate = n_before - len(df_merged)

# Remove negative dimensions
n_before2 = len(df_merged)
df_merged = df_merged[(df_merged['w'] > 0) & (df_merged['h'] > 0)].copy()
n_negative = n_before2 - len(df_merged)

# Flag deciduous teeth (keep them, don't remove)
df_merged['is_deciduous'] = df_merged['tooth_id'].isin(DECIDUOUS_IDS)
n_deciduous = df_merged['is_deciduous'].sum()

print(f"\n[Clean] Bbox filtering:")
print(f"  Removed degenerate (dim < {MIN_BBOX_DIM}px): {n_degenerate}")
print(f"  Removed negative dims: {n_negative}")
print(f"  Flagged deciduous (A-T): {n_deciduous} (KEPT for analysis)")
print(f"  Remaining: {len(df_merged)} tooth records from {df_merged['stem'].nunique()} images")

# Show mask distribution in working set
n_from_empty = df_merged[df_merged['mask_empty']]['stem'].nunique()
n_from_annotated = df_merged[~df_merged['mask_empty']]['stem'].nunique()
print(f"\n  Images with empty mask (all crops → normal):     {n_from_empty}")
print(f"  Images with annotated mask (per-crop overlap):    {n_from_annotated}")

---
## Cell 7 — Core Functions: Crop, Overlap, Jitter

These are the building blocks used in the main construction loop.

In [ ]:
def compute_expanded_bbox(x, y, w, h, img_w, img_h, fw=EXPAND_W, fh=EXPAND_H):
    """
    Asymmetric expanded bounding box: 1.5× W, 2.0× H.
    
    WHY asymmetric:
      Teeth have mean aspect ratio ~0.35 (3× taller than wide).
      Horizontal over-expansion → bleeds into adjacent teeth.
      Vertical expansion → captures alveolar bone below root.
    
    Returns: (cx1, cy1, cx2, cy2) clipped to image bounds.
    """
    cx = x + w / 2.0
    cy = y + h / 2.0
    nw = w * fw
    nh = h * fh

    cx1 = max(0, int(cx - nw / 2.0))
    cy1 = max(0, int(cy - nh / 2.0))
    cx2 = min(img_w, int(cx + nw / 2.0))
    cy2 = min(img_h, int(cy + nh / 2.0))

    return cx1, cy1, cx2, cy2


def compute_overlap_ratio(mask, cx1, cy1, cx2, cy2):
    """
    Pixel-overlap ratio for label assignment.
    
    overlap = (nonzero mask pixels inside crop) / (crop area)
    
    WHY NOT IoU:
      Mask blobs are tiny (sparse lesion annotations).
      Crop bboxes are large (crown + root + bone).
      IoU = intersection / union ≈ tiny / large ≈ ~0
      Even true positives → IoU near zero → false negatives.
    
    EDGE CASES:
      - One blob spanning two teeth → both get positive (correct)
      - Multiple blobs in one crop → all pixels counted (correct)
      - Blob barely clipping edge → low overlap, threshold decides
    """
    mask_crop = mask[cy1:cy2, cx1:cx2]

    if mask_crop.size == 0:
        return 0.0

    mask_pixels = np.count_nonzero(mask_crop > 0)
    crop_area = mask_crop.shape[0] * mask_crop.shape[1]

    if crop_area == 0:
        return 0.0

    return mask_pixels / crop_area


def apply_bbox_jitter(x, y, w, h, img_w, img_h, jitter_frac=JITTER_FRACTION):
    """
    Random ±15% perturbation to simulate YOLO detection noise.
    
    Without this: model trains on perfect GT bboxes but receives
    noisy YOLO bboxes at inference → distribution mismatch.
    """
    dx = random.uniform(-jitter_frac, jitter_frac) * w
    dy = random.uniform(-jitter_frac, jitter_frac) * h
    dw = random.uniform(-jitter_frac, jitter_frac) * w
    dh = random.uniform(-jitter_frac, jitter_frac) * h

    new_x = max(0, x + dx)
    new_y = max(0, y + dy)
    new_w = max(MIN_BBOX_DIM, w + dw)
    new_h = max(MIN_BBOX_DIM, h + dh)

    # Clip to image bounds
    if new_x + new_w > img_w:
        new_w = img_w - new_x
    if new_y + new_h > img_h:
        new_h = img_h - new_y

    return new_x, new_y, new_w, new_h


def extract_crop(img, cx1, cy1, cx2, cy2, target_size=CROP_SIZE):
    """Extract region and resize to 224×224."""
    crop = img[cy1:cy2, cx1:cx2]
    if crop.size == 0:
        return None
    return cv2.resize(crop, target_size, interpolation=cv2.INTER_LINEAR)


print("Core functions defined.")

---
## Cell 8 — Build Dataset (Main Loop)

For each tooth in each fully-annotated image:
1. Load radiograph + expert mask
2. Compute expanded crop coordinates
3. Compute pixel-overlap ratio
4. Assign binary label
5. Extract + resize crop
6. Generate jittered augmented copy

**Label assignment logic:**
- Empty mask image → ALL crops = 0 (normal)
- Annotated mask image → per-crop overlap > threshold → 1, else 0
- No mask available → excluded (already filtered)

In [ ]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

all_crops = []       # (crop_array, label, metadata_dict)
overlap_scores = []  # For threshold analysis
skipped_radio = 0
skipped_mask = 0
skipped_crop = 0

grouped = df_merged.groupby('stem')
n_images = len(grouped)

print(f"Processing {n_images} images...\n")

for idx, (stem, group) in enumerate(grouped):
    if (idx + 1) % 50 == 0 or idx == 0:
        print(f"  [{idx+1}/{n_images}] Processing {stem}...")

    # --- Load radiograph ---
    radio_path = None
    for ext in ['.JPG', '.jpg', '.png', '.PNG', '.jpeg']:
        candidate = os.path.join(RADIO_DIR, stem + ext)
        if os.path.exists(candidate):
            radio_path = candidate
            break

    if radio_path is None:
        skipped_radio += 1
        continue

    img = cv2.imread(radio_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        skipped_radio += 1
        continue

    img_h, img_w = img.shape[:2]

    # --- Load expert mask ---
    mask_path = group.iloc[0]['mask_path']
    mask_empty = group.iloc[0]['mask_empty']

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        skipped_mask += 1
        continue

    # --- Process each tooth ---
    for _, row in group.iterrows():
        tooth_id = row['tooth_id']
        x, y, w, h = row['x'], row['y'], row['w'], row['h']
        is_deciduous = row['is_deciduous']

        # -- ORIGINAL CROP --
        cx1, cy1, cx2, cy2 = compute_expanded_bbox(x, y, w, h, img_w, img_h)

        if (cx2 - cx1) < MIN_BBOX_DIM or (cy2 - cy1) < MIN_BBOX_DIM:
            skipped_crop += 1
            continue

        # Label assignment
        if mask_empty:
            label = 0
            overlap = 0.0
        else:
            overlap = compute_overlap_ratio(mask, cx1, cy1, cx2, cy2)
            label = 1 if overlap > OVERLAP_THRESHOLD else 0

        # Store overlap for analysis
        overlap_scores.append({
            'stem': stem, 'tooth_id': tooth_id,
            'overlap': overlap, 'label': label,
            'mask_empty': mask_empty, 'is_deciduous': is_deciduous,
        })

        # Extract crop
        crop = extract_crop(img, cx1, cy1, cx2, cy2)
        if crop is None:
            skipped_crop += 1
            continue

        meta = {
            'stem': stem, 'tooth_id': tooth_id,
            'overlap': overlap, 'is_deciduous': is_deciduous,
            'jittered': False,
            'cx1': cx1, 'cy1': cy1, 'cx2': cx2, 'cy2': cy2,
        }
        all_crops.append((crop, label, meta))

        # -- JITTERED CROP (augmentation) --
        jit_x, jit_y, jit_w, jit_h = apply_bbox_jitter(x, y, w, h, img_w, img_h)
        jcx1, jcy1, jcx2, jcy2 = compute_expanded_bbox(
            jit_x, jit_y, jit_w, jit_h, img_w, img_h
        )

        if (jcx2 - jcx1) >= MIN_BBOX_DIM and (jcy2 - jcy1) >= MIN_BBOX_DIM:
            jit_crop = extract_crop(img, jcx1, jcy1, jcx2, jcy2)
            if jit_crop is not None:
                # Re-compute overlap for jittered region
                if mask_empty:
                    jit_label = 0
                    jit_overlap = 0.0
                else:
                    jit_overlap = compute_overlap_ratio(mask, jcx1, jcy1, jcx2, jcy2)
                    jit_label = 1 if jit_overlap > OVERLAP_THRESHOLD else 0

                jit_meta = {
                    'stem': stem, 'tooth_id': tooth_id,
                    'overlap': jit_overlap, 'is_deciduous': is_deciduous,
                    'jittered': True,
                    'cx1': jcx1, 'cy1': jcy1, 'cx2': jcx2, 'cy2': jcy2,
                }
                all_crops.append((jit_crop, jit_label, jit_meta))

# --- Summary ---
labels = [c[1] for c in all_crops]
label_counts = Counter(labels)

print(f"\n{'='*50}")
print(f"DATASET CONSTRUCTION COMPLETE")
print(f"{'='*50}")
print(f"  Total crops:          {len(all_crops)}")
print(f"  Class 0 (normal):     {label_counts.get(0, 0)}")
print(f"  Class 1 (diseased):   {label_counts.get(1, 0)}")
if label_counts.get(1, 0) > 0:
    ratio = label_counts.get(0, 0) / label_counts.get(1, 0)
    print(f"  Imbalance ratio:      {ratio:.2f}x")
print(f"  Jittered copies:      {sum(1 for _,_,m in all_crops if m['jittered'])}")
print(f"  Deciduous teeth:      {sum(1 for _,_,m in all_crops if m['is_deciduous'])}")
print(f"  Skipped (no radio):   {skipped_radio}")
print(f"  Skipped (no mask):    {skipped_mask}")
print(f"  Skipped (bad crop):   {skipped_crop}")

df_overlaps = pd.DataFrame(overlap_scores)

---
## Cell 9 — Overlap Threshold Analysis

**Before committing to any threshold**, visualize the overlap distribution.  
This answers: *"What happens at overlap = 0.003? Is it noise, early lesion, or labeling artifact?"*

Your threshold decision **defines your ground truth**.

In [ ]:
# Only analyze annotated images (mask not empty)
df_ann = df_overlaps[~df_overlaps['mask_empty']].copy()

print(f"Overlap analysis — annotated images only ({len(df_ann)} tooth crops)\n")
print(f"  Crops with overlap > 0:       {(df_ann['overlap'] > 0).sum()}")
print(f"  Crops with overlap > 0.001:   {(df_ann['overlap'] > 0.001).sum()}")
print(f"  Crops with overlap > 0.005:   {(df_ann['overlap'] > 0.005).sum()}")
print(f"  Crops with overlap > 0.01:    {(df_ann['overlap'] > 0.01).sum()}")
print(f"  Crops with overlap > 0.02:    {(df_ann['overlap'] > 0.02).sum()}")
print(f"  Crops with overlap > 0.05:    {(df_ann['overlap'] > 0.05).sum()}")

nonzero = df_ann[df_ann['overlap'] > 0]['overlap']
if len(nonzero) > 0:
    print(f"\n  Nonzero overlap stats:")
    print(f"    Min:    {nonzero.min():.6f}")
    print(f"    Median: {nonzero.median():.6f}")
    print(f"    Mean:   {nonzero.mean():.6f}")
    print(f"    Max:    {nonzero.max():.6f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Full distribution
axes[0].hist(df_ann['overlap'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(OVERLAP_THRESHOLD, color='red', ls='--', lw=2, label=f'T={OVERLAP_THRESHOLD}')
axes[0].set_xlabel('Overlap ratio')
axes[0].set_ylabel('Count')
axes[0].set_title('All overlap scores (annotated images)')
axes[0].legend()

# (b) Zoomed: overlap < 0.05
low = df_ann[df_ann['overlap'] < 0.05]['overlap']
axes[1].hist(low, bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].axvline(OVERLAP_THRESHOLD, color='red', ls='--', lw=2, label=f'T={OVERLAP_THRESHOLD}')
axes[1].set_xlabel('Overlap ratio')
axes[1].set_title('Zoomed: overlap < 0.05 (borderline zone)')
axes[1].legend()

# (c) Log scale (nonzero only)
if len(nonzero) > 0:
    axes[2].hist(np.log10(nonzero), bins=50, color='seagreen', edgecolor='black', alpha=0.7)
    axes[2].axvline(np.log10(OVERLAP_THRESHOLD), color='red', ls='--', lw=2,
                    label=f'log10({OVERLAP_THRESHOLD})')
    axes[2].set_xlabel('log10(overlap)')
    axes[2].set_title('Log-scale (nonzero only)')
    axes[2].legend()

plt.suptitle(f'Overlap Threshold Analysis — Current T = {OVERLAP_THRESHOLD}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n⚠️  Study this plot. If the threshold cuts through a cluster of genuine")
print("   positives, adjust OVERLAP_THRESHOLD in Cell 2 and rerun from Cell 8.")

---
## Cell 10 — Stratified Train/Val Split (Image-Level)

**Split at image level**, not crop level.  
All crops from the same radiograph (including jittered copies) stay in the same split.  
Otherwise → data leakage → inflated metrics.

In [ ]:
# Group crops by source image
image_crops = defaultdict(list)
image_labels = {}

for crop, label, meta in all_crops:
    key = meta['stem']
    image_crops[key].append((crop, label, meta))
    # Image-level label: positive if ANY crop is positive
    if key not in image_labels:
        image_labels[key] = 0
    if label == 1:
        image_labels[key] = 1

stems = list(image_crops.keys())
img_labels = [image_labels[s] for s in stems]

# Stratified split
train_stems, val_stems = train_test_split(
    stems, test_size=VAL_RATIO, random_state=RANDOM_SEED, stratify=img_labels
)

train_crops = []
val_crops = []
for s in train_stems:
    train_crops.extend(image_crops[s])
for s in val_stems:
    val_crops.extend(image_crops[s])

train_labels = Counter([c[1] for c in train_crops])
val_labels   = Counter([c[1] for c in val_crops])

print(f"Image-level stratified split:")
print(f"  Train images: {len(train_stems)} | Val images: {len(val_stems)}")
print(f"  Train — Class 0: {train_labels.get(0,0)}, Class 1: {train_labels.get(1,0)}")
print(f"  Val   — Class 0: {val_labels.get(0,0)}, Class 1: {val_labels.get(1,0)}")

# Leakage check
leakage = set(train_stems) & set(val_stems)
print(f"\n  Leakage check: {'✓ CLEAN' if len(leakage)==0 else f'✗ {len(leakage)} LEAKED'}")

---
## Cell 11 — Save Dataset to Disk

```
processed_dataset_tufts/
  train/
    0/   ← normal
    1/   ← diseased
  val/
    0/
    1/
  metadata/
    train_metadata.csv
    val_metadata.csv
```

In [ ]:
# Clean output directory
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

for split in ['train', 'val']:
    for cls in ['0', '1']:
        os.makedirs(os.path.join(OUTPUT_DIR, split, cls), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'metadata'), exist_ok=True)


def save_split(crops, split_name):
    metadata_rows = []
    counters = {0: 0, 1: 0}

    for crop, label, meta in crops:
        idx = counters[label]
        fname = f"{meta['stem']}_T{meta['tooth_id']}"
        if meta['jittered']:
            fname += "_jit"
        fname += f"_{idx:04d}.png"

        fpath = os.path.join(OUTPUT_DIR, split_name, str(label), fname)
        cv2.imwrite(fpath, crop)

        metadata_rows.append({
            'filename': fname, 'split': split_name, 'label': label,
            'stem': meta['stem'], 'tooth_id': meta['tooth_id'],
            'overlap': meta['overlap'], 'is_deciduous': meta['is_deciduous'],
            'jittered': meta['jittered'],
        })
        counters[label] += 1

    df_meta = pd.DataFrame(metadata_rows)
    csv_path = os.path.join(OUTPUT_DIR, 'metadata', f'{split_name}_metadata.csv')
    df_meta.to_csv(csv_path, index=False)

    print(f"  [{split_name}] Class 0: {counters[0]} | Class 1: {counters[1]} | Total: {sum(counters.values())}")
    return df_meta


print(f"Saving dataset to: {OUTPUT_DIR}\n")
df_train_meta = save_split(train_crops, 'train')
df_val_meta   = save_split(val_crops, 'val')

print(f"\n✓ Dataset saved.")

---
## Cell 12 — Visual Validation (NON-NEGOTIABLE)

**If you skip this, your model learns garbage.**

For 30 samples across three categories:
- **True positives** (high overlap, label=1)
- **Borderline negatives** (low nonzero overlap, label=0)
- **Clean negatives** (empty mask, label=0)

Each shows: crop + red mask overlay + assigned label + overlap score.  
If labels don't match visible pathology → **adjust threshold and rebuild**.

In [ ]:
# Categorize non-jittered crops
positives        = [(c,l,m) for c,l,m in all_crops if l==1 and not m['jittered']]
neg_annotated    = [(c,l,m) for c,l,m in all_crops if l==0 and not m['jittered'] and m['overlap']>0]
neg_empty        = [(c,l,m) for c,l,m in all_crops if l==0 and not m['jittered'] and m['overlap']==0]

print(f"Sample pool:")
print(f"  Positives (label=1):           {len(positives)}")
print(f"  Borderline negatives (overlap>0, label=0): {len(neg_annotated)}")
print(f"  Clean negatives (empty mask):  {len(neg_empty)}")

# Select samples
random.seed(RANDOM_SEED)
n_pos = min(N_VISUAL_SAMPLES // 3, len(positives))
n_neg_ann = min(N_VISUAL_SAMPLES // 3, len(neg_annotated))
n_neg_emp = min(N_VISUAL_SAMPLES - n_pos - n_neg_ann, len(neg_empty))

samples = []
if n_pos > 0:     samples += random.sample(positives, n_pos)
if n_neg_ann > 0: samples += random.sample(neg_annotated, n_neg_ann)
if n_neg_emp > 0: samples += random.sample(neg_empty, n_neg_emp)

print(f"\nSelected: {n_pos} positive + {n_neg_ann} borderline + {n_neg_emp} clean = {len(samples)} total")

In [ ]:
n_cols = 5
n_rows = max(1, (len(samples) + n_cols - 1) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 4*n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flatten()

for i, (crop, label, meta) in enumerate(samples):
    if i >= len(axes):
        break

    ax = axes[i]

    # Load mask for overlay
    mask_row = df_merged[df_merged['stem'] == meta['stem']].iloc[0]
    mask = cv2.imread(mask_row['mask_path'], cv2.IMREAD_GRAYSCALE)

    cx1, cy1, cx2, cy2 = meta['cx1'], meta['cy1'], meta['cx2'], meta['cy2']
    mask_crop = mask[cy1:cy2, cx1:cx2]
    mask_resized = cv2.resize(mask_crop, CROP_SIZE, interpolation=cv2.INTER_NEAREST)

    # Overlay: grayscale crop + red mask
    overlay = cv2.cvtColor(crop, cv2.COLOR_GRAY2RGB)
    mask_bin = (mask_resized > 0)
    overlay[mask_bin, 0] = np.minimum(overlay[mask_bin, 0].astype(int) + 120, 255).astype(np.uint8)
    overlay[mask_bin, 1] = (overlay[mask_bin, 1] * 0.4).astype(np.uint8)
    overlay[mask_bin, 2] = (overlay[mask_bin, 2] * 0.4).astype(np.uint8)

    ax.imshow(overlay)

    label_str = "DISEASED" if label == 1 else "NORMAL"
    color = 'red' if label == 1 else 'lime'
    dec = " [DEC]" if meta['is_deciduous'] else ""

    ax.set_title(
        f"{meta['stem']} T{meta['tooth_id']}{dec}\n"
        f"{label_str} | overlap={meta['overlap']:.4f}",
        fontsize=8, color=color, fontweight='bold'
    )
    ax.axis('off')

# Hide unused
for j in range(len(samples), len(axes)):
    axes[j].axis('off')

plt.suptitle(
    f'VISUAL VALIDATION — Threshold: {OVERLAP_THRESHOLD}\n'
    f'Red = expert pathology mask | Verify labels match actual pathology',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'visual_validation.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n⚠️  REVIEW THE PLOT ABOVE CAREFULLY.")
print("   If labels don't match visible pathology:")
print("   → Adjust OVERLAP_THRESHOLD in Cell 2")
print("   → Rerun from Cell 8")

---
## Cell 13 — Dataset Integrity Checks

Systematic checks before you trust this dataset for training.

In [ ]:
print("=" * 60)
print("DATASET INTEGRITY CHECKS")
print("=" * 60)

# 1. Class balance
print(f"\n[1] Class balance:")
for name, df in [('Train', df_train_meta), ('Val', df_val_meta)]:
    counts = df['label'].value_counts()
    total = len(df)
    print(f"  {name}: Class 0 = {counts.get(0,0)} ({counts.get(0,0)/total*100:.1f}%), "
          f"Class 1 = {counts.get(1,0)} ({counts.get(1,0)/total*100:.1f}%)")

# 2. Image leakage
train_s = set(df_train_meta['stem'])
val_s = set(df_val_meta['stem'])
leakage = train_s & val_s
status = '✓ NONE' if len(leakage) == 0 else f'✗ {len(leakage)} LEAKED'
print(f"\n[2] Image leakage: {status}")

# 3. Deciduous teeth
print(f"\n[3] Deciduous teeth:")
for name, df in [('Train', df_train_meta), ('Val', df_val_meta)]:
    n_dec = df['is_deciduous'].sum()
    print(f"  {name}: {n_dec} ({n_dec/len(df)*100:.1f}%)")

# 4. Regional bias — are all positives from same few images?
print(f"\n[4] Regional bias (positive concentration):")
for name, df in [('Train', df_train_meta), ('Val', df_val_meta)]:
    pos = df[df['label'] == 1]
    if len(pos) > 0:
        n_src = pos['stem'].nunique()
        top = pos['stem'].value_counts().iloc[0]
        print(f"  {name}: {len(pos)} positives from {n_src} images (max {top} from one image)")
    else:
        print(f"  {name}: ⚠️  No positive samples!")

# 5. Overlap scores per class
print(f"\n[5] Overlap scores by label:")
for name, df in [('Train', df_train_meta), ('Val', df_val_meta)]:
    for cls in [0, 1]:
        sub = df[df['label'] == cls]['overlap']
        if len(sub) > 0:
            print(f"  {name} Class {cls}: mean={sub.mean():.5f}, "
                  f"min={sub.min():.5f}, max={sub.max():.5f}")

# 6. Verify files on disk
print(f"\n[6] Files on disk:")
for split in ['train', 'val']:
    for cls in ['0', '1']:
        folder = os.path.join(OUTPUT_DIR, split, cls)
        n_files = len(os.listdir(folder))
        print(f"  {split}/{cls}/: {n_files} files")

print(f"\n{'='*60}")
print(f"ALL CHECKS COMPLETE")
print(f"{'='*60}")

---
## Cell 14 — Sample Grid: Positive vs Negative

Quick 4×4 grid showing random samples from each class.  
Confirms that crops look correct and classes are visually distinguishable.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(24, 6))

# Top row: Class 0 (normal)
normal_dir = os.path.join(OUTPUT_DIR, 'train', '0')
normal_files = os.listdir(normal_dir)
random.seed(RANDOM_SEED)
normal_sample = random.sample(normal_files, min(8, len(normal_files)))

for i, fname in enumerate(normal_sample):
    img = cv2.imread(os.path.join(normal_dir, fname), cv2.IMREAD_GRAYSCALE)
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(f'Normal', fontsize=9, color='lime')
    axes[0, i].axis('off')

# Bottom row: Class 1 (diseased)
disease_dir = os.path.join(OUTPUT_DIR, 'train', '1')
disease_files = os.listdir(disease_dir)
disease_sample = random.sample(disease_files, min(8, len(disease_files)))

for i, fname in enumerate(disease_sample):
    img = cv2.imread(os.path.join(disease_dir, fname), cv2.IMREAD_GRAYSCALE)
    axes[1, i].imshow(img, cmap='gray')
    axes[1, i].set_title(f'Diseased', fontsize=9, color='red')
    axes[1, i].axis('off')

# Hide unused
for row in range(2):
    for col in range(len(normal_sample if row==0 else disease_sample), 8):
        axes[row, col].axis('off')

plt.suptitle('Sample Grid: Normal (top) vs Diseased (bottom)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_grid.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 15 — Final Summary

Print everything needed to proceed to model training.

In [ ]:
print("\n" + "=" * 60)
print("✓ DATASET CONSTRUCTION PIPELINE COMPLETE")
print("=" * 60)
print(f"\nDataset location: {OUTPUT_DIR}")
print(f"\nStructure:")
print(f"  train/0/ (normal):   {len(os.listdir(os.path.join(OUTPUT_DIR,'train','0')))} crops")
print(f"  train/1/ (diseased): {len(os.listdir(os.path.join(OUTPUT_DIR,'train','1')))} crops")
print(f"  val/0/   (normal):   {len(os.listdir(os.path.join(OUTPUT_DIR,'val','0')))} crops")
print(f"  val/1/   (diseased): {len(os.listdir(os.path.join(OUTPUT_DIR,'val','1')))} crops")
print(f"\nConfig used:")
print(f"  Overlap threshold: {OVERLAP_THRESHOLD}")
print(f"  Crop expansion:    {EXPAND_W}x W, {EXPAND_H}x H")
print(f"  Bbox jitter:       ±{JITTER_FRACTION*100:.0f}%")
print(f"  Crop size:         {CROP_SIZE}")
print(f"  Val ratio:         {VAL_RATIO}")
print(f"  Random seed:       {RANDOM_SEED}")
print(f"\nSaved artifacts:")
print(f"  {OUTPUT_DIR}/metadata/train_metadata.csv")
print(f"  {OUTPUT_DIR}/metadata/val_metadata.csv")
print(f"  {OUTPUT_DIR}/overlap_analysis.png")
print(f"  {OUTPUT_DIR}/visual_validation.png")
print(f"  {OUTPUT_DIR}/sample_grid.png")
print(f"\n" + "=" * 60)
print(f"NEXT STEP: If visual validation passes →")
print(f"  Train ResNet18 binary classifier on this dataset")
print(f"  with weighted loss and F1-based evaluation.")
print(f"\nIf visual validation FAILS →")
print(f"  Adjust OVERLAP_THRESHOLD in Cell 2 and rerun from Cell 8.")
print("=" * 60)